# Quali Position Model - Exploration

In [2]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from xgboost import XGBRegressor
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

from app.config import (
    PROCESSED_HISTORIC_FEATURES_DIR, PROCESSED_PRACTICE_FEATURES_DIR,
    INTERIM_QUALI_DIR, ARTIFACTS_DIR,
    TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
)
from app.models.configs import QUALI_POSITION_MODEL

## Load data

In [3]:
historic_features = pd.concat([pd.read_parquet(f) for f in sorted(PROCESSED_HISTORIC_FEATURES_DIR.glob('*.parquet'))])
practice_features = pd.concat([pd.read_parquet(f) for f in sorted(PROCESSED_PRACTICE_FEATURES_DIR.glob('*.parquet'))])
quali_results = pd.concat([pd.read_parquet(f) for f in sorted(INTERIM_QUALI_DIR.glob('*.parquet'))])

df = historic_features.merge(
    practice_features,
    on=['race_id', 'driver_id'],
    how='left'
).merge(
    quali_results[['race_id', 'driver_id', 'quali_position']],
    on=['race_id', 'driver_id'],
    how='left'
)

df = df.dropna(subset=['quali_position'])

print(df.shape)
df.tail()

(3454, 32)


,race_id,driver_id,rolling_quali_pos_last_3,rolling_quali_pos_last_5,rolling_finish_pos_last_3,rolling_finish_pos_last_5,rolling_fantasy_points_last_3,rolling_fantasy_points_last_5,rolling_dnf_rate_last_5,circuit_rolling_quali_pos_last_3,...,constructor_form_trend_last_5,fp2_gap_to_leader_pct,fp3_gap_to_leader_pct,fp3_sector1_gap_to_leader_pct,fp3_sector2_gap_to_leader_pct,fp3_sector3_gap_to_leader_pct,fp3_teammate_gap_pct,fp2_longrun_avg_gap_to_field_pct,fp2_laps_completed,quali_position
3453,2025_24,alexander_albon,14.333333,15.8,12.666667,12.8,-5.333333,-6.4,0.4,16.333333,...,11.2,1.043535,0.465596,0.405239,0.196774,1.895941,-0.106191,1.494461,32.0,17.0
3454,2025_24,isack_hadjar,6.333333,9.6,10.666667,12.2,1.666667,-2.8,0.4,NaN,...,2.9,0.690875,0.773994,0.816351,0.501635,2.153414,0.038119,-11.427398,29.0,9.0
3455,2025_24,liam_lawson,8.333333,10.4,10.000000,12.2,0.000000,-3.8,0.4,12.000000,...,2.9,1.674229,0.735594,0.880954,0.629123,1.855815,-0.038105,3.043327,32.0,13.0
3456,2025_24,pierre_gasly,9.333333,12.0,13.000000,14.6,-6.000000,-11.6,0.6,11.000000,...,3.2,2.262797,0.885593,1.074764,0.382462,2.461045,-0.507686,4.058652,30.0,19.0
3457,2025_24,franco_colapinto,17.666667,17.6,14.666667,15.4,-13.000000,-15.8,0.8,19.000000,...,3.2,2.031703,1.400389,1.585717,0.911812,2.925834,0.510277,-3.286428,28.0,20.0


## Train/val/test split

In [4]:
config = QUALI_POSITION_MODEL

X = df[config['features']]
y = df[config['target']]

X_train = X[df['season'].isin(TRAIN_SEASONS)]
y_train = y[df['season'].isin(TRAIN_SEASONS)]

X_val = X[df['season'].isin(VAL_SEASONS)]
y_val = y[df['season'].isin(VAL_SEASONS)]

X_test = X[df['season'].isin(TEST_SEASONS)]
y_test = y[df['season'].isin(TEST_SEASONS)]

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

Train: 2497 | Val: 479 | Test: 478


## Baselines

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np

def evaluate(model, X, y, split_name):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    spearman = spearmanr(y, y_pred).statistic
    print(f'[{split_name}] MAE: {mae:.2f} | Spearman: {spearman:.3f}')

def evaluate_naive(y, y_pred, split_name):
    mask = ~np.isnan(y_pred)
    mae = mean_absolute_error(y[mask], y_pred[mask])
    spearman = spearmanr(y[mask], y_pred[mask]).statistic
    print(f'  [{split_name}] MAE: {mae:.2f} | Spearman: {spearman:.3f}')

# naive: predict rolling quali position as quali position
naive_val_pred = X_val['rolling_quali_pos_last_3'].values
naive_test_pred = X_test['rolling_quali_pos_last_3'].values
print('Naive (rolling quali pos last 3):')
evaluate_naive(y_val.values, naive_val_pred, 'val')
evaluate_naive(y_test.values, naive_test_pred, 'test')

# linear regression
lr = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('model', LinearRegression())])
lr.fit(X_train, y_train)
print('\nLinear Regression:')
print(f'  [val]  MAE: {mean_absolute_error(y_val, lr.predict(X_val)):.2f} | Spearman: {spearmanr(y_val, lr.predict(X_val)).statistic:.3f}')
print(f'  [test] MAE: {mean_absolute_error(y_test, lr.predict(X_test)):.2f} | Spearman: {spearmanr(y_test, lr.predict(X_test)).statistic:.3f}')

# random forest
rf = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('model', RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))])
rf.fit(X_train, y_train)
print('\nRandom Forest:')
print(f'  [val]  MAE: {mean_absolute_error(y_val, rf.predict(X_val)):.2f} | Spearman: {spearmanr(y_val, rf.predict(X_val)).statistic:.3f}')
print(f'  [test] MAE: {mean_absolute_error(y_test, rf.predict(X_test)):.2f} | Spearman: {spearmanr(y_test, rf.predict(X_test)).statistic:.3f}')

Naive (rolling quali pos last 3):
  [val] MAE: 3.21 | Spearman: 0.684
  [test] MAE: 3.30 | Spearman: 0.684

Linear Regression:
  [val]  MAE: 3.03 | Spearman: 0.717
  [test] MAE: 3.20 | Spearman: 0.697

Random Forest:
  [val]  MAE: 2.96 | Spearman: 0.745
  [test] MAE: 3.20 | Spearman: 0.711


## Baseline: no practice features

In [6]:
baseline_features = [f for f in config['features'] if not f.startswith('fp')]
X_train_base = X_train[baseline_features]
X_val_base = X_val[baseline_features]
X_test_base = X_test[baseline_features]

baseline_model = XGBRegressor(**config['hyperparams'])
baseline_model.fit(X_train_base, y_train, eval_set=[(X_val_base, y_val)], verbose=False)

print('Baseline (no practice):')
evaluate(baseline_model, X_val_base, y_val, 'val')
evaluate(baseline_model, X_test_base, y_test, 'test')

Baseline (no practice):
[val] MAE: 3.06 | Spearman: 0.712
[test] MAE: 3.30 | Spearman: 0.682


## XGBoost model

In [7]:
model = XGBRegressor(**config['hyperparams'])
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print('XGBoost:')
evaluate(model, X_val, y_val, 'val')
evaluate(model, X_test, y_test, 'test')

XGBoost:
[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715


## Feature importance

In [8]:
importance = pd.Series(model.feature_importances_, index=config['features']).sort_values(ascending=False)
print(importance.head(config['feature_importance_top_n']))

rolling_quali_pos_last_5                0.287578
constructor_rolling_quali_pos_last_3    0.274024
rolling_quali_pos_last_3                0.106273
rolling_finish_pos_last_5               0.094169
season_points_to_date                   0.037571
fp3_gap_to_leader_pct                   0.034244
fp2_gap_to_leader_pct                   0.032863
fp3_sector2_gap_to_leader_pct           0.024539
rolling_finish_pos_last_3               0.013339
fp3_sector1_gap_to_leader_pct           0.012490
dtype: float32


## Hyperparameter tuning

In [9]:
# experiment with hyperparams here
tuned_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=20,
)

tuned_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

evaluate(tuned_model, X_val, y_val, 'val')
evaluate(tuned_model, X_test, y_test, 'test')

[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715


#### Tune max_depth

In [10]:
for depth in [3, 4, 5, 6]:
    m = XGBRegressor(**{**config['hyperparams'], 'max_depth': depth})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'max_depth={depth}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

max_depth=3:
[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715
max_depth=4:
[val] MAE: 2.88 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.711
max_depth=5:
[val] MAE: 2.93 | Spearman: 0.752
[test] MAE: 3.26 | Spearman: 0.700
max_depth=6:
[val] MAE: 2.92 | Spearman: 0.745
[test] MAE: 3.26 | Spearman: 0.698


#### Tune learning_rate

In [11]:
for lr in [0.01, 0.05, 0.1, 0.2]:
    m = XGBRegressor(**{**config['hyperparams'], 'learning_rate': lr})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'learning_rate={lr}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

learning_rate=0.01:
[val] MAE: 3.52 | Spearman: 0.735
[test] MAE: 3.67 | Spearman: 0.711
learning_rate=0.05:
[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715
learning_rate=0.1:
[val] MAE: 2.88 | Spearman: 0.753
[test] MAE: 3.19 | Spearman: 0.709
learning_rate=0.2:
[val] MAE: 2.89 | Spearman: 0.751
[test] MAE: 3.20 | Spearman: 0.711


#### Tune n_estimators

In [12]:
for n in [100, 200, 300, 500]:
    m = XGBRegressor(**{**config['hyperparams'], 'n_estimators': n})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'n_estimators={n}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

n_estimators=100:
[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715
n_estimators=200:
[val] MAE: 2.85 | Spearman: 0.755
[test] MAE: 3.17 | Spearman: 0.715
n_estimators=300:
[val] MAE: 2.85 | Spearman: 0.755
[test] MAE: 3.17 | Spearman: 0.715
n_estimators=500:
[val] MAE: 2.85 | Spearman: 0.755
[test] MAE: 3.17 | Spearman: 0.715


#### Tune subsample & colsample_bytree

In [13]:
for sub, col in [(0.6, 0.6), (0.7, 0.7), (0.8, 0.8), (0.9, 0.9), (1.0, 1.0)]:
    m = XGBRegressor(**{**config['hyperparams'], 'subsample': sub, 'colsample_bytree': col})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'subsample={sub}, colsample_bytree={col}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

subsample=0.6, colsample_bytree=0.6:
[val] MAE: 2.88 | Spearman: 0.750
[test] MAE: 3.15 | Spearman: 0.720
subsample=0.7, colsample_bytree=0.7:
[val] MAE: 2.87 | Spearman: 0.756
[test] MAE: 3.16 | Spearman: 0.717
subsample=0.8, colsample_bytree=0.8:
[val] MAE: 2.87 | Spearman: 0.755
[test] MAE: 3.18 | Spearman: 0.715
subsample=0.9, colsample_bytree=0.9:
[val] MAE: 2.87 | Spearman: 0.754
[test] MAE: 3.17 | Spearman: 0.714
subsample=1.0, colsample_bytree=1.0:
[val] MAE: 2.89 | Spearman: 0.749
[test] MAE: 3.18 | Spearman: 0.714
